# Finer-level ZNCC refinement of the coarse peak (iteration 28)

**Why.** The coarse score map is one value per 4-pixel cell. At level 10 (1300 m) a pixel is 9.6 m, a cell 38 m, while sigma is about 21 m, so the peak sits on a grid about 1.8 sigma wide: a quantisation error of about 0.5 sigma alone, and the answers measured 1.2-2.2 sigma off above 800 m. Averaging queries could not remove it (iteration 27).

**Variants, per query (coarse peak of that query).**
- `c`: the coarse peak as it is (baseline).
- `p`: parabola through the score at the peak and its 4 neighbours (sub-cell offset in x and y, clipped to +-0.5 cell). No extra compute.
- `f2`, `f4`: re-run the ZNCC on the map two / four levels finer (pixel 2x / 4x smaller, cell 2x / 4x smaller), only in a window of +-4 coarse cells around the peak, then the same parabola (`f2p`, `f4p`). The canvas is redrawn at the finer level, so it covers a smaller centre part of the photo.
Photo level = the answer of the query with the highest coarse z (as in iterations 25-27). Error in sigma against the truth from metadata.

**Kill rule (written before the run).** Useful only if on the new set above 800 m the share within 1 sigma rises clearly (from 24.6 % to at least 40 %) with no new wrong answers (beyond 8 sigma), and on the old official set and NUDT it does not lose points.

Env: `CR_HEADLESS, CR_DATASET, CR_STORE, CR_LEVELS, CR_TAG, CR_FIRST, CR_LIMIT, CR_JITTER, CR_OUT, CR_GEOMETRY`.

In [ ]:
import os, sys, math
import numpy as np, pandas as pd, torch
HEADLESS = os.environ.get("CR_HEADLESS", "../../../../Main_Pipeline_Headless")
sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("CR_LEVELS", "3,4,5,6,7").split(",")]
for item in os.environ.get("CR_GEOMETRY", "").split(","):
    if item:
        key, value = item.split("="); settings.GEOMETRY[key] = (value == "True") if value in ("True", "False") else float(value)
from geoloc import photo, queries, scoring
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query
from geoloc.levels import level_for_height, level_gsd
from geoloc.map_store import MapStore
DATASET = os.environ["CR_DATASET"]; STORE = os.environ["CR_STORE"]
TAG = os.environ.get("CR_TAG", "run"); FIRST = int(os.environ.get("CR_FIRST", 0)); LIMIT = int(os.environ.get("CR_LIMIT", 0))
JITTER = float(os.environ.get("CR_JITTER", 5000)); OUT = os.environ.get("CR_OUT", "results/coarse_refine"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)[FIRST:]
if LIMIT: rows = rows[:LIMIT]
print(DEVICE, len(rows), "photos, levels", settings.LEVELS, "store", sorted(store.levels))

In [ ]:
def best_cell(score_map, geometry):
    usable = (score_map > -1) & geometry["inside"]
    ranking = torch.where(usable, score_map, torch.full_like(score_map, -1e9))
    value, flat = ranking.reshape(-1).max(0)
    cols = score_map.shape[1]; return int(flat) // cols, int(flat) % cols, float(value)

def parabola(score_map, row, col):
    # sub-cell offsets in cells from the neighbours; 0 where a neighbour is unusable
    def offset(left, centre, right):
        if min(left, right) <= -1: return 0.0
        curve = left - 2 * centre + right
        return 0.0 if curve >= -1e-9 else float(np.clip(0.5 * (left - right) / curve, -0.5, 0.5))
    h, w = score_map.shape; c = float(score_map[row, col])
    dx = offset(float(score_map[row, max(col-1, 0)]), c, float(score_map[row, min(col+1, w-1)]))
    dy = offset(float(score_map[max(row-1, 0), col]), c, float(score_map[min(row+1, h-1), col]))
    return dx, dy

def peak_and_parabola(score_map, geometry, level):
    row, col, value = best_cell(score_map, geometry)
    x, y = float(geometry["x_m"][col]), float(geometry["y_m"][row])
    dx, dy = parabola(score_map, row, col); step = settings.CELL * level_gsd(level)
    return (x, y), (x + dx * step, y + dy * step), value

def disc_z(score_map, geometry, value):
    usable = (score_map > -1) & geometry["inside"]; v = score_map[usable]
    return (value - float(v.mean())) / float(v.std())

In [ ]:
records = []; base_radius = settings.DISC_RADIUS; query_list = queries.query_list()
for index, meta in enumerate(rows):
    meta = move_hint(meta, index + FIRST, JITTER, 0) if JITTER > 0 else meta
    camera = photo.read_camera(meta); pose = photo.read_pose(meta)
    prior = store.gps_to_metres(float(meta["lat_sc"]), float(meta["lon_sc"])); truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"]))
    sigma = scoring.sigma_m(pose["height"], camera["fx"]); level = level_for_height(pose["height"], camera)
    path = os.path.join(DATASET, "images", meta["filename"])
    pictures = {level: photo.prepare_photo(path, pose, camera, level)}
    for q, query in enumerate(query_list):
        settings.DISC_RADIUS = base_radius
        try:
            canvas, valid = photo.draw_canvas(pictures[level], pose, camera, level, query["heading"], query["scale_factor"])
            score_map, geometry = score_map_for_query(canvas, valid, level, prior, store.features, {})
        except ValueError:
            continue
        coarse, para, value = peak_and_parabola(score_map, geometry, level)
        row = dict(photo=index + FIRST, name=meta["filename"], q=q, height=pose["height"], sigma_m=sigma, level=level, z1=disc_z(score_map, geometry, value), tx=truth[0], ty=truth[1])
        variants = {"c": coarse, "p": para}
        for drop in (2, 4):
            fine = level - drop
            if fine < min(settings.LEVELS) or fine not in store.levels: continue
            try:
                if fine not in pictures: pictures[fine] = photo.prepare_photo(path, pose, camera, fine)
                settings.DISC_RADIUS = 4 * settings.CELL * level_gsd(level)
                canvas_f, valid_f = photo.draw_canvas(pictures[fine], pose, camera, fine, query["heading"], query["scale_factor"])
                sm_f, geo_f = score_map_for_query(canvas_f, valid_f, fine, para, store.features, {})
                fc, fp, _ = peak_and_parabola(sm_f, geo_f, fine)
                variants["f%d" % drop] = fc; variants["f%dp" % drop] = fp
            except (ValueError, RuntimeError):
                pass
        for name, (x, y) in variants.items():
            row["x_" + name] = x; row["y_" + name] = y; row["err_" + name] = math.hypot(x - truth[0], y - truth[1]) / sigma
        records.append(row)
    settings.DISC_RADIUS = base_radius
    if index % 20 == 0:
        print(index + 1, "/", len(rows), flush=True); pd.DataFrame(records).to_csv(os.path.join(OUT, "refine_%s.csv" % TAG), index=False)
table = pd.DataFrame(records); table.to_csv(os.path.join(OUT, "refine_%s.csv" % TAG), index=False); print(len(table), "query rows saved")

## Photo level: the query with the highest coarse z, error in sigma per variant

In [ ]:
def pts(e): return np.where(e<=1,10,np.where(e<=2,6,np.where(e<=4,3,np.where(e<=8,0,-20))))
best = table.loc[table.groupby("photo").z1.idxmax()]
variants = [c[4:] for c in table.columns if c.startswith("err_")]
def summary(frame, label):
    out = []
    for v in variants:
        e = frame["err_" + v].dropna()
        out.append(dict(variant=v, n=len(e), within1=round((e<=1).mean(),3), within2=round((e<=2).mean(),3), wrong=int((e>8).sum()), median_not_wrong=round(e[e<=8].median(),2), pts_per_photo=round(pts(e.values).mean(),2)))
    print("\n==", label); print(pd.DataFrame(out).to_string(index=False))
summary(best, "all photos"); summary(best[best.height >= 800], "above 800 m")
for lo, hi in [(0,500),(500,800),(800,1300),(1300,2200)]: summary(best[(best.height>=lo)&(best.height<hi)], "%d-%d m" % (lo, hi))